# Human annotation agreement and dynamic multimodal errors

Reproduce the annotation-agreement paragraph in **Section 6.3 of the submitted TAFFC article**. This notebook uses external CSV files only; it contains no embedded data snapshots.

Keep this notebook in `6_discussion/6_3_interpreting_persistent_error_patterns/`, beside `multimodal_fea_analysis.ipynb`. That notebook produces the FEA profiles for supplemental Figures S6–S7; this one computes the complementary human-agreement results. Run all cells in order. Dependencies: Python, pandas, NumPy, and IPython/Jupyter.

The analysis retains only:

- 2/3 versus 3/3 agreement counts in the held-out test set;
- multimodal error rates and the share of errors on 2/3-agreement cases;
- the accuracy difference after equal weighting of the six classes present in both groups;
- agreement between erroneous multimodal predictions and the dissenting rater's label.

No figures are required for these prose results. Small supporting tables, input checksums, and a Markdown report are exported to `outputs/label_agreement_and_model_errors/`. Dataset-wide membership analyses, Image/FEA comparisons, correlations, shuffle benchmarks, bootstrap intervals, and sensitivity evaluations are omitted.

**Interpretation:** three raters judged one selected HMD-occluded central-view reference image per reenactment. Its retained label is inherited by the sequences. Agreement characterizes that image; it does not establish objective correctness of the full sequence, side view, or FEA signal. The 756 view predictions represent 378 paired reenactments, not 756 independent annotation cases. All calculations here are descriptive and do not relabel samples.


## 1. External inputs and paths

- Predictions: `5_dynamic_facial_expression_recognition/dynamic_test_predictions.csv`, shared with the other dynamic analyses. Only its multimodal predictions are used.
- Annotations: `labels-master-table-analysis_irr.csv` within Section 6.3. Keep the **complete, unchanged, semicolon-delimited upstream CSV** (2,590 rows); no restructuring or test-only extraction is needed.
- Annotation source: [pinned upstream CSV](https://github.com/thorbenortmann/emoji-hero-vr-database/blob/deee4796cbdbfbd39ce70e99c7adb8836feee86e/v_data_annotation/b_results/labels-master-table-analysis_irr.csv), commit `deee4796cbdbfbd39ce70e99c7adb8836feee86e`. See `data/annotation_source.md` for provenance. The local copy makes execution independent of network availability and future upstream changes.

The CSV column names are adapted in memory. Only annotations matched to the shared test predictions enter the reported calculations. Retention and reference-image checks prevent accidental inclusion of other annotated candidates.

Run from the repository root or a directory beneath it. `REPO_ROOT` may be set explicitly. The output directory is anchored to Section 6.3, independent of the notebook's working directory.


In [1]:
from pathlib import Path
import hashlib
import io

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

REPO_ROOT = None  # Optional: Path('/absolute/path/to/emohevrdb-dfer')

def find_repo_root(explicit=None):
    if explicit is not None:
        candidates = [Path(explicit).expanduser().resolve()]
    else:
        cwd = Path.cwd().resolve()
        candidates = [cwd, *cwd.parents, cwd / 'emohevrdb-dfer']
    for candidate in candidates:
        if (candidate / '5_dynamic_facial_expression_recognition/dynamic_test_predictions.csv').is_file():
            return candidate
    raise FileNotFoundError('Set REPO_ROOT to the repository containing 5_dynamic_facial_expression_recognition/dynamic_test_predictions.csv.')

REPO_ROOT = find_repo_root(REPO_ROOT)
SECTION_DIR = REPO_ROOT / '6_discussion/6_3_interpreting_persistent_error_patterns'
PREDICTIONS_PATH = REPO_ROOT / '5_dynamic_facial_expression_recognition/dynamic_test_predictions.csv'
ANNOTATIONS_PATH = SECTION_DIR / 'labels-master-table-analysis_irr.csv'
OUTPUT_DIR = SECTION_DIR / 'outputs/label_agreement_and_model_errors'
CATEGORIES = ['Anger', 'Disgust', 'Fear', 'Happiness', 'Neutral', 'Sadness', 'Surprise']
ANNOTATION_SHA256 = 'f06ae02ac6f4118e27558fa08303eadcabc9f90b01ae851867c1a1f9cf894d19'
ANNOTATION_SOURCE = 'https://github.com/thorbenortmann/emoji-hero-vr-database/blob/deee4796cbdbfbd39ce70e99c7adb8836feee86e/v_data_annotation/b_results/labels-master-table-analysis_irr.csv'

def require(condition, message):
    if not bool(condition):
        raise ValueError(message)

if not ANNOTATIONS_PATH.is_file():
    raise FileNotFoundError(f'Place the unchanged upstream annotation CSV at {ANNOTATIONS_PATH}. Source: {ANNOTATION_SOURCE}')

annotation_bytes = ANNOTATIONS_PATH.read_bytes()
prediction_bytes = PREDICTIONS_PATH.read_bytes()
require(hashlib.sha256(annotation_bytes).hexdigest() == ANNOTATION_SHA256,
        'Annotation checksum differs from the pinned source. Restore the unchanged CSV; document and verify any intentional source update before changing ANNOTATION_SHA256.')
labels = pd.read_csv(io.BytesIO(annotation_bytes), sep=';', dtype={'image': str})
predictions = pd.read_csv(io.BytesIO(prediction_bytes), dtype={'timestamp': str})
provenance = pd.DataFrame([
    {'input': 'annotations', 'local_path': str(ANNOTATIONS_PATH), 'source': ANNOTATION_SOURCE,
     'sha256': hashlib.sha256(annotation_bytes).hexdigest(), 'rows': len(labels)},
    {'input': 'dynamic_test_predictions', 'local_path': str(PREDICTIONS_PATH),
     'source': 'shared Section 5 prediction export',
     'sha256': hashlib.sha256(prediction_bytes).hexdigest(), 'rows': len(predictions)},
])
display(provenance[['input', 'rows', 'sha256']])


,input,rows,sha256
0,annotations,2590,f06ae02ac6f4118e27558fa08303eadcabc9f90b01ae85...
1,dynamic_test_predictions,756,ac322c3232bd9a8bcb46f1be049233c2260b3193ebd191...


## 2. Validate and join annotations

Join on participant, level, and emoji identifiers, then verify the annotation's reference-image filename against the prediction timestamp and its prompted category against the benchmark label. Recompute agreement from the three individual labels rather than trusting the stored aggregate alone.

Both views receive the same annotation record. Their multimodal predictions may differ. Reenactment counts use one row per reenactment; model error rates use both view rows, matching the paper's evaluation unit.


In [2]:
keys = ['participant_id', 'level_id', 'emoji_id']
ratings = ['label_1', 'label_2', 'label_3']
annotation_columns = ['participant', 'level', 'emoji', 'image', 'target_emotion',
                      *ratings, 'agreement', 'include_in_dataset']
prediction_columns = ['sample_id', 'reenactment_id', 'timestamp', *keys,
                      'camera_index', 'perspective', 'true_label_id', 'true_label',
                      'multimodal_pred_id', 'multimodal_pred']
for frame, columns, name in [(labels, annotation_columns, 'annotations'),
                             (predictions, prediction_columns, 'predictions')]:
    require(set(columns).issubset(frame.columns), f'Missing {name} columns: {sorted(set(columns) - set(frame.columns))}')
    require(frame[columns].notna().all().all(), f'Missing values in required {name} columns.')

require(len(labels) == 2590, 'The pinned annotation table should contain 2,590 rows.')
require(set(labels[ratings + ['target_emotion']].to_numpy().ravel()).issubset(CATEGORIES),
        'Unknown annotation category.')
recomputed_agreement = labels[ratings].eq(labels['target_emotion'], axis=0).sum(axis=1)
require(recomputed_agreement.eq(labels['agreement']).all(), 'Stored agreement does not match individual ratings.')
require(labels['include_in_dataset'].eq(recomputed_agreement.ge(2).astype(int)).all(),
        'Annotation inclusion flag differs from the two-of-three criterion.')
labels = labels.rename(columns={'participant': 'participant_id', 'level': 'level_id', 'emoji': 'emoji_id'})
require(not labels.duplicated(keys).any(), 'Duplicate annotation identifiers.')

predictions = predictions[prediction_columns].copy()
require(predictions['sample_id'].is_unique, 'Duplicate sample IDs.')
require(not predictions.duplicated(['reenactment_id', 'camera_index']).any(), 'Duplicate view rows.')
require(predictions.groupby('reenactment_id').size().eq(2).all(), 'Each reenactment must contain two views.')
require(predictions.groupby('reenactment_id')['camera_index'].apply(set).map(lambda x: x == {0, 1}).all(), 'Unexpected camera indices.')
require(predictions['camera_index'].map({0: 'Central', 1: 'Side'}).eq(predictions['perspective']).all(), 'Camera/perspective mismatch.')
stable_columns = [*keys, 'timestamp', 'true_label_id', 'true_label']
require(predictions.groupby('reenactment_id')[stable_columns].nunique().eq(1).all().all(), 'Inconsistent reenactment metadata.')
require(not predictions.drop_duplicates('reenactment_id').duplicated(keys).any(), 'Annotation key maps to multiple reenactments.')
id_to_label = dict(enumerate(CATEGORIES))
for id_column, label_column in [('true_label_id', 'true_label'), ('multimodal_pred_id', 'multimodal_pred')]:
    require(predictions[id_column].map(id_to_label).eq(predictions[label_column]).all(), f'Category ID/name mismatch: {label_column}.')

view_data = predictions.merge(labels[keys + ['image', 'target_emotion', *ratings, 'agreement', 'include_in_dataset']],
                              on=keys, how='left', validate='many_to_one', indicator=True)
require(view_data['_merge'].eq('both').all(), 'Some predictions lack an annotation record.')
require(view_data['true_label'].eq(view_data['target_emotion']).all(), 'Annotation/prediction target mismatch.')
require((view_data['timestamp'] + '.png').eq(view_data['image']).all(), 'Reference-image timestamp mismatch.')
require(view_data['include_in_dataset'].eq(1).all(), 'Predictions include an excluded annotation candidate.')
require(view_data['agreement'].isin([2, 3]).all(), 'Retained test samples must have 2/3 or 3/3 agreement.')
view_data = view_data.drop(columns='_merge')
view_data['correct'] = view_data['multimodal_pred'].eq(view_data['true_label'])
view_data['minority_label'] = view_data.apply(
    lambda row: next((row[c] for c in ratings if row[c] != row['true_label']), None)
    if row['agreement'] == 2 else None, axis=1)
view_data['minority_match'] = view_data['agreement'].eq(2) & view_data['multimodal_pred'].eq(view_data['minority_label'])
reenactments = view_data.drop_duplicates('reenactment_id')
require(len(view_data) == 756 and len(reenactments) == 378,
        'Expected the submitted benchmark: 756 views from 378 reenactments.')
require(reenactments['participant_id'].nunique() == 8, 'Expected eight test participants.')
require(view_data.groupby('true_label').size().reindex(CATEGORIES).eq(108).all(), 'Expected 108 test views per category.')
print(f'Validated {len(view_data)} view predictions, {len(reenactments)} reenactments, and eight participants.')


Validated 756 view predictions, 378 reenactments, and eight participants.


## 3. Agreement-group counts and multimodal errors

Compute the fraction of test reenactments with 2/3 agreement, the error rates within each agreement group, and each group's contribution to all multimodal errors. The paired views have identical annotations, so the group proportion is the same at the reenactment and view levels.


In [3]:
total_errors = int((~view_data['correct']).sum())
require(total_errors > 0, 'No model errors to analyze.')
summary_rows = []
for agreement in [2, 3]:
    group = view_data.loc[view_data['agreement'].eq(agreement)]
    n_reenactments = int(reenactments['agreement'].eq(agreement).sum())
    require(len(group) > 0, f'No {agreement}/3 samples.')
    errors = int((~group['correct']).sum())
    summary_rows.append({
        'agreement': f'{agreement}/3', 'reenactments': n_reenactments,
        'reenactment_share_percent': 100 * n_reenactments / len(reenactments),
        'view_predictions': len(group), 'correct': int(group['correct'].sum()), 'errors': errors,
        'accuracy_percent': 100 * group['correct'].mean(),
        'error_rate_percent': 100 * errors / len(group),
        'share_of_all_errors_percent': 100 * errors / total_errors,
    })
agreement_summary = pd.DataFrame(summary_rows)
display(agreement_summary.round(2))


,agreement,reenactments,reenactment_share_percent,view_predictions,correct,errors,accuracy_percent,error_rate_percent,share_of_all_errors_percent
0,2/3,103,27.25,206,144,62,69.9,30.1,44.6
1,3/3,275,72.75,550,473,77,86.0,14.0,55.4


## 4. Equal class weighting

The groups differ in class composition. Happiness has no 2/3-agreement test samples. Compute recall within every class and agreement group, then average recalls equally over the **six classes represented in both groups**. The difference is 3/3 accuracy minus 2/3 accuracy, in percentage points. Classes with zero support remain missing rather than receiving a zero recall. This is a descriptive class-composition adjustment, not participant balancing or a causal estimate.


In [4]:
class_rows = []
for category in CATEGORIES:
    for agreement in [2, 3]:
        group = view_data.loc[view_data['true_label'].eq(category) & view_data['agreement'].eq(agreement)]
        correct = int(group['correct'].sum())
        class_rows.append({'category': category, 'agreement': f'{agreement}/3',
                           'reenactments': group['reenactment_id'].nunique(),
                           'view_predictions': len(group), 'correct': correct, 'errors': len(group) - correct,
                           'recall_percent': 100 * correct / len(group) if len(group) else np.nan})
class_summary = pd.DataFrame(class_rows)
recall = class_summary.pivot(index='category', columns='agreement', values='recall_percent').reindex(CATEGORIES)
common_recall = recall.dropna(subset=['2/3', '3/3'])
require(len(common_recall) == 6 and 'Happiness' not in common_recall.index,
        'The shared-class composition differs from the submitted analysis.')
equal_class = common_recall.mean()
group_index = agreement_summary.set_index('agreement')
raw_gap_pp = float(group_index.loc['3/3', 'accuracy_percent'] - group_index.loc['2/3', 'accuracy_percent'])
adjusted_gap_pp = float(equal_class['3/3'] - equal_class['2/3'])
standardized_summary = pd.DataFrame([{
    'shared_classes': len(common_recall), 'categories': ' | '.join(common_recall.index),
    'equal_class_accuracy_2_of_3_percent': equal_class['2/3'],
    'equal_class_accuracy_3_of_3_percent': equal_class['3/3'],
    'raw_advantage_3_minus_2_pp': raw_gap_pp,
    'equal_class_advantage_3_minus_2_pp': adjusted_gap_pp,
}])
display(recall.round(2))
display(standardized_summary.round(2))


agreement,2/3,3/3
category,,
Anger,39.29,61.25
Disgust,59.62,73.21
Fear,84.29,76.32
Happiness,NaN,100.00
Neutral,92.86,94.68
Sadness,85.71,93.62
Surprise,64.29,86.25


,shared_classes,categories,equal_class_accuracy_2_of_3_percent,equal_class_accuracy_3_of_3_percent,raw_advantage_3_minus_2_pp,equal_class_advantage_3_minus_2_pp
0,6,Anger | Disgust | Fear | Neutral | Sadness | S...,71.01,80.89,16.1,9.88


## 5. Dissenting-label correspondence

Among 2/3-agreement cases, exactly one rater disagreed with the retained category. Count how often an **erroneous view prediction** equals that rater's alternative label. Both perspectives contribute separately to the denominator, as in the paper. This correspondence does not establish that either alternative label is objectively correct.


In [5]:
two_vote_errors = view_data.loc[view_data['agreement'].eq(2) & ~view_data['correct']]
require(len(two_vote_errors) > 0, 'No errors on 2/3-agreement cases.')
minority_matches = int(two_vote_errors['minority_match'].sum())
minority_summary = pd.DataFrame([{
    'agreement': '2/3', 'error_view_predictions': len(two_vote_errors),
    'dissenting_label_matches': minority_matches,
    'dissenting_label_match_percent': 100 * minority_matches / len(two_vote_errors),
}])
display(minority_summary.round(2))


,agreement,error_view_predictions,dissenting_label_matches,dissenting_label_match_percent
0,2/3,62,49,79.03


## 6. Export the paper results

All prose numbers below are computed from the current files. Rounded expectations from the submitted manuscript are checked separately; they are not substituted for calculated results. If predictions change, the status column flags changed paper values rather than silently retaining old numbers.

Exports: four supporting CSV tables, a paper-number check table, an input-provenance CSV, and a concise Markdown report. There are no generated figures or embedded output datasets.


In [6]:
paper_numbers = pd.DataFrame([
    {'claim': '2/3 share of test reenactments', 'unit': 'percent',
     'computed': group_index.loc['2/3', 'reenactment_share_percent'], 'paper': 27.25},
    {'claim': '2/3 share of all multimodal errors', 'unit': 'percent',
     'computed': group_index.loc['2/3', 'share_of_all_errors_percent'], 'paper': 44.60},
    {'claim': '2/3 multimodal error rate', 'unit': 'percent',
     'computed': group_index.loc['2/3', 'error_rate_percent'], 'paper': 30.10},
    {'claim': '3/3 multimodal error rate', 'unit': 'percent',
     'computed': group_index.loc['3/3', 'error_rate_percent'], 'paper': 14.00},
    {'claim': 'Raw 3/3 accuracy advantage', 'unit': 'percentage points',
     'computed': raw_gap_pp, 'paper': 16.10},
    {'claim': 'Equal-class 3/3 accuracy advantage', 'unit': 'percentage points',
     'computed': adjusted_gap_pp, 'paper': 9.88},
    {'claim': 'Multimodal errors on 2/3 cases', 'unit': 'view predictions',
     'computed': len(two_vote_errors), 'paper': 62},
    {'claim': 'Dissenting-label matches among these errors', 'unit': 'view predictions',
     'computed': minority_matches, 'paper': 49},
    {'claim': 'Dissenting-label match rate', 'unit': 'percent',
     'computed': minority_summary.loc[0, 'dissenting_label_match_percent'], 'paper': 79.03},
])
paper_numbers['matches_paper_rounding'] = np.isclose(paper_numbers['computed'].round(2), paper_numbers['paper'], atol=1e-9, rtol=0)
display(paper_numbers.round(2))
if not paper_numbers['matches_paper_rounding'].all():
    print('Some values differ from the submitted paper. Review the prediction provenance and input versions.')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
tables = {
    'agreement_group_summary.csv': agreement_summary,
    'class_recall_by_agreement.csv': class_summary,
    'equal_class_accuracy_summary.csv': standardized_summary,
    'dissenting_label_correspondence.csv': minority_summary,
    'paper_number_checks.csv': paper_numbers,
    'input_provenance.csv': provenance,
}
for filename, table in tables.items():
    table.to_csv(OUTPUT_DIR / filename, index=False, float_format='%.10f')

a2 = group_index.loc['2/3']
a3 = group_index.loc['3/3']
report = f'''# Section 6.3: human annotation agreement and multimodal errors

Test data: {len(view_data)} view predictions from {len(reenactments)} reenactments and eight participants; {total_errors} multimodal errors.

- 2/3 agreement: {int(a2['reenactments'])}/{len(reenactments)} reenactments ({a2['reenactment_share_percent']:.2f}%).
- Errors on 2/3 cases: {int(a2['errors'])}/{total_errors} of all errors ({a2['share_of_all_errors_percent']:.2f}%).
- Error rates: {int(a2['errors'])}/{int(a2['view_predictions'])} = {a2['error_rate_percent']:.2f}% for 2/3, and {int(a3['errors'])}/{int(a3['view_predictions'])} = {a3['error_rate_percent']:.2f}% for 3/3.
- Raw 3/3 accuracy advantage: {raw_gap_pp:.2f} percentage points.
- Equal weighting of {len(common_recall)} shared classes: {equal_class['2/3']:.2f}% accuracy for 2/3, {equal_class['3/3']:.2f}% for 3/3, a {adjusted_gap_pp:.2f}-percentage-point advantage. Happiness is excluded because it has no 2/3 test cases.
- Dissenting-label correspondence: {minority_matches}/{len(two_vote_errors)} erroneous 2/3-agreement view predictions ({minority_summary.loc[0, 'dissenting_label_match_percent']:.2f}%).
- All numbers match the submitted paper at reported precision: {bool(paper_numbers['matches_paper_rounding'].all())}.

Annotation source: {ANNOTATION_SOURCE}

Annotation SHA256: {hashlib.sha256(annotation_bytes).hexdigest()}

Prediction SHA256: {hashlib.sha256(prediction_bytes).hexdigest()}

The complete unchanged annotation CSV is filtered by joining to the shared test predictions. The three raters judged one selected central-view reference image per reenactment; both camera views inherit those annotations. These are descriptive associations, with no inferential tests, confidence intervals, or label corrections. Equal class weighting controls only class composition; it does not establish a causal agreement effect.

Supporting tables: {', '.join(tables)}.
'''
(OUTPUT_DIR / 'label_agreement_and_model_errors_report.md').write_text(report, encoding='utf-8')
display(Markdown(report))
print(f'Wrote six CSV tables and one report to {OUTPUT_DIR}')


,claim,unit,computed,paper,matches_paper_rounding
0,2/3 share of test reenactments,percent,27.25,27.25,True
1,2/3 share of all multimodal errors,percent,44.60,44.60,True
2,2/3 multimodal error rate,percent,30.10,30.10,True
3,3/3 multimodal error rate,percent,14.00,14.00,True
4,Raw 3/3 accuracy advantage,percentage points,16.10,16.10,True
5,Equal-class 3/3 accuracy advantage,percentage points,9.88,9.88,True
6,Multimodal errors on 2/3 cases,view predictions,62.00,62.00,True
7,Dissenting-label matches among these errors,view predictions,49.00,49.00,True
8,Dissenting-label match rate,percent,79.03,79.03,True


# Section 6.3: human annotation agreement and multimodal errors

Test data: 756 view predictions from 378 reenactments and eight participants; 139 multimodal errors.

- 2/3 agreement: 103/378 reenactments (27.25%).
- Errors on 2/3 cases: 62/139 of all errors (44.60%).
- Error rates: 62/206 = 30.10% for 2/3, and 77/550 = 14.00% for 3/3.
- Raw 3/3 accuracy advantage: 16.10 percentage points.
- Equal weighting of 6 shared classes: 71.01% accuracy for 2/3, 80.89% for 3/3, a 9.88-percentage-point advantage. Happiness is excluded because it has no 2/3 test cases.
- Dissenting-label correspondence: 49/62 erroneous 2/3-agreement view predictions (79.03%).
- All numbers match the submitted paper at reported precision: True.

Annotation source: https://github.com/thorbenortmann/emoji-hero-vr-database/blob/deee4796cbdbfbd39ce70e99c7adb8836feee86e/v_data_annotation/b_results/labels-master-table-analysis_irr.csv

Annotation SHA256: f06ae02ac6f4118e27558fa08303eadcabc9f90b01ae851867c1a1f9cf894d19

Prediction SHA256: ac322c3232bd9a8bcb46f1be049233c2260b3193ebd19197c421ea372e007cde

The complete unchanged annotation CSV is filtered by joining to the shared test predictions. The three raters judged one selected central-view reference image per reenactment; both camera views inherit those annotations. These are descriptive associations, with no inferential tests, confidence intervals, or label corrections. Equal class weighting controls only class composition; it does not establish a causal agreement effect.

Supporting tables: agreement_group_summary.csv, class_recall_by_agreement.csv, equal_class_accuracy_summary.csv, dissenting_label_correspondence.csv, paper_number_checks.csv, input_provenance.csv.


Wrote six CSV tables and one report to /workspace/repos/emohevrdb-dfer/6_discussion/6_3_interpreting_persistent_error_patterns/outputs/label_agreement_and_model_errors
